# Clase 09 — Ejercicio intergrador Clínica MediCare

> Notebook de práctica. Ejecutá las celdas a medida que avanzás en el artículo.

La clínica MediCare lleva varios años registrando turnos médicos en un sistema interno. El sistema fue cambiando de versión con el tiempo, y los datos se fueron exportando a una base de datos SQLite sin demasiado control de calidad.

El director médico necesita un informe con algunas preguntas concretas sobre el funcionamiento de la clínica, pero antes de poder responderlas, alguien tiene que hacer la limpieza. Ese trabajo es el de ustedes, no se les dice qué problemas hay en los datos. Parte del ejercicio es encontrarlos.

> El archivo de la base de datos `medicare.db` ya está disponible para que lo uses en este ejercicio.

### 1) Exploración: ¿qué tiene esta base de datos?

Antes de tocar nada, el primer trabajo es entender qué hay adentro. Conectense a `medicare.db` y respondan estas preguntas.

1. ¿Qué tablas tiene la base de datos?
2. ¿Qué columnas y tipos tiene cada tabla?
3. ¿Cuántas filas tiene cada tabla?

In [20]:
# tu codigo aca
import sqlite3
import pandas as pd

conexion = sqlite3.connect('medicare.db')

df = pd.read_sql(
    """
    SELECT *
    FROM sqlite_master
    """,
    conexion
) 

#print (df) #1 ¿Qué tablas tiene la base de datos?

df = pd.read_sql(
    """
    PRAGMA table_info(pacientes)

    """,
    conexion
)

#print (df) #2 ¿Qué columnas y tipos tiene cada tabla?

df = pd.read_sql(
    """
    PRAGMA table_info(medicos)

    """,
    conexion
)

#print (df) #2

df = pd.read_sql(
    """
    PRAGMA table_info(turnos)

    """,
    conexion
)

#print (df) #2

df = pd.read_sql(
    """
    SELECT COUNT (*)
    FROM pacientes
    """,
    conexion
)

print(df)#3 ¿Cuántas filas tiene cada tabla?


   COUNT (*)
0         66


**Documenten sus hallazgos.** Antes de seguir al Paso 2, escriban en comentarios qué encontraron.
La BDD tiene 3 tablas , para saber la estructura de cada tabla se uso pragma (table_info() ) por ultimo se conto la cantidad de registros de cada tabla usando COUNT(*)

### 2) Diagnóstico: ¿qué problemas tienen los datos?

Traigan las tres tablas a DataFrames y hagan un diagnóstico completo de cada una. Para cada tabla respondan:

- ¿Cuántos nulos hay por columna? ¿Qué porcentaje representan?
- ¿Hay filas duplicadas? ¿Cuáles?
- ¿Hay columnas con tipos incorrectos que van a impedir operar con los datos?
- ¿Hay valores que parecen errores aunque no sean nulos? (Pista: revisen la columna `costo` en `turnos`)


In [54]:
# tu codigo aquí

df_pacientes = pd.read_sql(
    """
    SELECT *
    FROM pacientes
    """,
    conexion
)

diagnostico_nulos_pacientes = pd.DataFrame({
    "Cantidad de nulos PACIENTES": df_pacientes.isnull().sum(),
    "Porcentaje de nulos PACIENTES": (df_pacientes.isnull().sum() / len(df_pacientes)) * 100
})

print(diagnostico_nulos_pacientes)

print("Cantidad de filas duplicadas exactas:")
print(df_pacientes.duplicated().sum())

print("Duplicados por DNI:")
print(df_pacientes.duplicated(subset=["dni"]).sum())

print("Tipos de datos:")
print(df_pacientes.dtypes)


# MEDICOS

df_medicos = pd.read_sql(
    """
    SELECT *
    FROM medicos
    """,
    conexion
)

diagnostico_nulos_medicos = pd.DataFrame({
    "Cantidad de nulos MEDICOS": df_medicos.isnull().sum(),
    "Porcentaje de nulos MEDICOS": (df_medicos.isnull().sum() / len(df_medicos)) * 100
})

print(diagnostico_nulos_medicos)

print("Cantidad de filas duplicadas exactas:")
print(df_medicos.duplicated().sum())

print("Duplicados por nombre:")
print(df_medicos.duplicated(subset=["nombre"]).sum())

print("Tipos de datos:")
print(df_medicos.dtypes)


# TURNOS

df_turnos = pd.read_sql(
    """
    SELECT *
    FROM turnos
    """,
    conexion
)

diagnostico_nulos_turnos = pd.DataFrame({
    "Cantidad de nulos TURNOS": df_turnos.isnull().sum(),
    "Porcentaje de nulos TURNOS": (df_turnos.isnull().sum() / len(df_turnos)) * 100
})

print(diagnostico_nulos_turnos)

print("Cantidad de filas duplicadas exactas:")
print(df_turnos.duplicated().sum())

print("Tipos de datos:")
print(df_turnos.dtypes)


# Valores problemáticos en costo

print("Valores de costo:")
print(df_turnos["costo"].unique())

                  Cantidad de nulos PACIENTES  Porcentaje de nulos PACIENTES
id                                          0                       0.000000
nombre                                      0                       0.000000
dni                                         7                      10.606061
fecha_nacimiento                           11                      16.666667
ciudad                                      8                      12.121212
obra_social                                12                      18.181818
Cantidad de filas duplicadas exactas:
0
Duplicados por DNI:
14
Tipos de datos:
id                  int64
nombre                str
dni                   str
fecha_nacimiento      str
ciudad                str
obra_social           str
dtype: object
              Cantidad de nulos MEDICOS  Porcentaje de nulos MEDICOS
id                                    0                          0.0
nombre                                0                          0.0
espec

**Antes de seguir al Paso 3, escriban una lista con todos los problemas que encontraron, uno por línea.** No empiecen a limpiar todavía.
 Problemas encontrados:
 - Hay valores nulos en varias columnas de la tabla pacientes.
 - La columna fecha_nacimiento está almacenada como texto.
 - La columna costo de la tabla turnos está almacenada como texto.
 - La columna costo contiene valores negativos, "N/A", cadenas vacías y valores nulos.

### 3) Decisiones: ¿qué hacemos con cada problema?

Este es el paso más importante del ejercicio. Para cada problema que encontraron en el Paso 2, decidan qué van a hacer y por qué. No hay una respuesta única correcta — lo que importa es que la decisión sea coherente con el análisis que necesitan hacer después.

Completen esta tabla antes de escribir código:

| Tabla | Columna | Problema | Decisión | Justificación |
|-------|---------|----------|----------|---------------|
| pacientes | dni | 7 nulo | ... | ... |
| pacientes | fecha de nacimiento | 11 nulo y tipo str | convertir | debe ser una fecha |
| pacientes | ciudad | 8 nulos | ... | ... |
| pacientes | obra social | 12 nulo | ... | ... |
| medicos | matricula | 2 nulo | ... | ... |
| medicos | activo | 4 nulo | ... | ... |
| turnos | costos | tipo str | convertir | debe ser numerico |
| turnos | costos | Valores inválidos (N/A, "", negativos) | ... | ... |






### 4) Limpieza: aplicar las decisiones

Ahora sí, implementen las decisiones del Paso 3. Recuerden:

- Primero eliminen duplicados
- Después manejen nulos
- Después conviertan tipos

In [28]:
# Limpieza de pacientes
df_pacientes_limpio = df_pacientes.copy()
# Tu código acá
# Eliminar duplicados
df_pacientes_limpio = df_pacientes_limpio.drop_duplicates()

# Convertir fecha_nacimiento a fecha
df_pacientes_limpio["fecha_nacimiento"] = pd.to_datetime(
    df_pacientes_limpio["fecha_nacimiento"],
    errors="coerce")


# Limpieza de médicos
df_medicos_limpio = df_medicos.copy()
# Tu código acá
# Eliminar duplicados
df_medicos_limpio = df_medicos_limpio.drop_duplicates()


# Limpieza de turnos
df_turnos_limpio = df_turnos.copy()
# Tu código acá
# Eliminar duplicados
df_turnos_limpio = df_turnos_limpio.drop_duplicates()

# Reemplazar valores inválidos por NaN
df_turnos_limpio["costo"] = df_turnos_limpio["costo"].replace(
    ["N/A", "", "-500", "-1000", "-3500"],
    pd.NA
)

# Convertir costo a número
df_turnos_limpio["costo"] = pd.to_numeric(
    df_turnos_limpio["costo"],
    errors="coerce"
)

> **Por qué `.copy()`:** cuando hacen `df_pacientes_limpio = df_pacientes`, no están creando una copia — están apuntando al mismo objeto. Cualquier cambio en `df_pacientes_limpio` modifica también `df_pacientes`. `.copy()` crea una copia independiente, así conservan los datos originales para comparar.

Al terminar, impriman un resumen de qué cambió en cada tabla:

In [34]:
# tu codigo aquí
#print (df_pacientes_limpio)
#print (df_medicos_limpio)
#print (df_turnos_limpio)

print("Pacientes")
print(df_pacientes.shape)
print(df_pacientes_limpio.shape)

print("\nMédicos")
print(df_medicos.shape)
print(df_medicos_limpio.shape)

print("\nTurnos")
print(df_turnos.shape)
print(df_turnos_limpio.shape)

Pacientes
(66, 6)
(66, 6)

Médicos
(20, 5)
(20, 5)

Turnos
(210, 8)
(210, 8)


### 5) — Análisis: responder las preguntas del director

Con los datos limpios, respondan estas cuatro preguntas. Para cada una, combinen SQL (para traer los datos necesarios) con Pandas (para calcular o presentar el resultado):

**Pregunta 1:** ¿Cuántos turnos realizó cada médico? Mostrá el resultado ordenado de mayor a menor.


**Pregunta 2:** ¿Cuál es el costo total facturado por especialidad médica? Considerá solo los turnos con estado `'realizado'`.

**Pregunta 3:** ¿Qué obra social tiene más pacientes registrados? ¿Y cuántos pacientes no tienen obra social?

**Pregunta 4:** ¿Cuántos turnos fueron cancelados o marcados como `'ausente'`? ¿Qué médico tuvo más cancelaciones?


In [ ]:
# Pregunta 1

consulta = """
SELECT
    m.nombre,
    COUNT(*) AS cantidad_turnos
FROM turnos t
JOIN medicos m
ON t.medico_id = m.id
GROUP BY m.nombre
ORDER BY cantidad_turnos DESC;
"""

turnos_por_medico = pd.read_sql(consulta, conexion)

turnos_por_medico

# Pregunta 2
consulta = """
SELECT
    m.especialidad,
    SUM(CAST(t.costo AS REAL)) AS costo_total
FROM turnos t
JOIN medicos m
ON t.medico_id = m.id
WHERE t.estado = 'realizado'
GROUP BY m.especialidad
ORDER BY costo_total DESC;
"""

facturacion_especialidad = pd.read_sql(consulta, conexion)

facturacion_especialidad

# Pregunta 3
consulta = """
SELECT
    obra_social
FROM pacientes;
"""

pacientes_obra = pd.read_sql(consulta, conexion)

pacientes_obra.head()

cantidad_por_obra = pacientes_obra["obra_social"].value_counts()

cantidad_por_obra

obra_mayor = cantidad_por_obra.idxmax()
cantidad_mayor = cantidad_por_obra.max()

print("Obra social con más pacientes:", obra_mayor)
print("Cantidad de pacientes:", cantidad_mayor)

sin_obra_social = pacientes_obra["obra_social"].isna().sum()

print("Pacientes sin obra social:", sin_obra_social)

#df_pacientes.columns

# Pregunta 4

consulta = """
SELECT
    estado,
    COUNT(*) AS cantidad
FROM turnos
WHERE estado IN ('cancelado', 'ausente')
GROUP BY estado;
"""

turnos_cancelados_ausentes = pd.read_sql(consulta, conexion)

turnos_cancelados_ausentes


consulta = """
SELECT
    m.nombre,
    COUNT(*) AS cantidad_cancelaciones
FROM turnos t
JOIN medicos m
ON t.medico_id = m.id
WHERE t.estado IN ('cancelado', 'ausente')
GROUP BY m.nombre
ORDER BY cantidad_cancelaciones DESC;
"""

cancelaciones_medico = pd.read_sql(consulta, conexion)

cancelaciones_medico

#ver resulñtado final
total_cancelados_ausentes = turnos_cancelados_ausentes["cantidad"].sum()

print("Total de turnos cancelados o ausentes:", total_cancelados_ausentes)



Obra social con más pacientes: Swiss Medical
Cantidad de pacientes: 14
Pacientes sin obra social: 12
Total de turnos cancelados o ausentes: 51


### 6) Reporte final

Escriban un bloque de código que imprima un reporte con los resultados de los cuatro puntos anteriores, en formato legible. No es necesario que sea elaborado — lo importante es que quede claro qué encontraron.

In [ ]:
print("=" * 50)
print("REPORTE CLÍNICA MEDICARE")
print("=" * 50)

# tu reporte acá

→ [Ir a la solución](./solucion.md)